# Run the whole avatar on Colab, and talk to it from your Mac

Colab has the GPU; your Mac has the microphone and the browser. This puts the server on
Colab and exposes it through a free HTTPS tunnel, so the page you open locally streams your
voice up and video back down.

**Run this today with the stub renderer, before M0 succeeds.** It proves the hosting path --
tunnel, WebSocket, microphone permission, HTTPS -- works. Finding out that a tunnel breaks
WebSockets *after* spending hours on the model would be the expensive order to discover it
in. When M0 and M2 land, the only change here is one line: `AVATAR_RENDERER=musetalk`.

**Runtime → Change runtime type → T4 GPU** before you start.

### Put your keys in Colab Secrets, not in this notebook

Left sidebar → the **key icon** → add each of these, and toggle *Notebook access* on:

| Name | Value |
|---|---|
| `DEEPGRAM_API_KEY` | your Deepgram key |
| `OPENAI_API_KEY` | your Ollama Cloud key |

Anything typed into a cell gets saved into the notebook file. Secrets do not.


## 1. Clone the repo

Colab needs the code from GitHub. If this fails with *Repository not found*, the push has
not happened yet -- see `docs/COLAB_HOSTING.md` step 0.


In [ ]:
import os, re, subprocess, sys, time
from pathlib import Path

REPO = 'https://github.com/prashanth-chinnala/nod.git'
WORK = Path('/content/nod')


def sh(cmd, quiet=False, check=False):
    if not quiet:
        print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout and not quiet:
        print(r.stdout[-2500:])
    if r.returncode != 0 and not quiet:
        print('STDERR:', (r.stderr or '')[-2500:], file=sys.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(cmd)
    return r


if not WORK.exists():
    r = sh(f'git clone --depth 1 {REPO} {WORK}')
    if r.returncode != 0:
        raise SystemExit(
            'Clone failed. If it says "Repository not found", the code is not on GitHub yet:\n'
            '  cd ~/nod && git push -u origin main\n'
            'A private repo also needs a token in the URL. See docs/COLAB_HOSTING.md step 0.'
        )
else:
    sh(f'cd {WORK} && git pull --ff-only', quiet=True)

os.chdir(WORK)
print('at', Path.cwd(), '-- commit', sh('git rev-parse --short HEAD', quiet=True).stdout.strip())


## 2. Install

`[server]` and `[tts]` only. No GPU packages here -- the renderer is still the stub. The
orchestration layer has no ML dependency at all, which is what makes this install take
seconds instead of minutes.


In [ ]:
sh('pip install -q -e ".[server,tts,llm]" 2>&1 | tail -5')
r = sh('python -m pytest -m "not gpu" -q 2>&1 | tail -3')
print('\nIf that says "199 passed", the code arrived intact.')


## 3. Configuration, from Colab Secrets


In [ ]:
from google.colab import userdata


def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


DEEPGRAM = secret('DEEPGRAM_API_KEY')
OPENAI = secret('OPENAI_API_KEY')

# Written to .env.development, which the server loads itself -- so no later cell
# needs an env prefix. Values never printed, and the file dies with the runtime.
lines = [
    'AVATAR_RENDERER=stub',        # <-- becomes musetalk once M0 and M2 land
    'AVATAR_VAD=energy',
]
if DEEPGRAM:
    lines += ['AVATAR_TTS=deepgram', 'AVATAR_STT=deepgram',
              f'DEEPGRAM_API_KEY={DEEPGRAM}', 'AVATAR_TTS_VOICE=aura-2-thalia-en']
else:
    lines += ['AVATAR_TTS=tone', 'AVATAR_STT=none']
    print('!! No DEEPGRAM_API_KEY secret -- falling back to the placeholder tone and no STT.')
if OPENAI:
    lines += ['AVATAR_LLM=openai', 'OPENAI_BASE_URL=https://ollama.com/v1',
              f'OPENAI_API_KEY={OPENAI}', 'AVATAR_LLM_MODEL=gpt-oss:20b']
else:
    lines += ['AVATAR_LLM=scripted']
    print('!! No OPENAI_API_KEY secret -- falling back to canned questions.')

Path('.env.development').write_text('\n'.join(lines) + '\n')
os.chmod('.env.development', 0o600)
print('\n.env.development written. Names only:')
print('  ' + ', '.join(sorted(line.split('=')[0] for line in lines)))


## 4. Start the server and open a public HTTPS tunnel

`cloudflared` gives a free `https://<random>.trycloudflare.com` URL with no account. Two
reasons it has to be HTTPS and not plain HTTP: browsers refuse microphone access outside a
secure context, and the page upgrades its WebSocket to `wss://` to match. A plain-HTTP
tunnel would load the page and then silently fail at the microphone.


In [ ]:
# cloudflared: free, no signup, and proxies WebSockets -- which this needs for both the
# video frames coming down and the microphone audio going up.
if not Path('/usr/local/bin/cloudflared').exists():
    sh('wget -q -O /tmp/cf.deb '
       'https://github.com/cloudflare/cloudflared/releases/latest/download/'
       'cloudflared-linux-amd64.deb && dpkg -i /tmp/cf.deb 2>&1 | tail -2')

sh('pkill -f "uvicorn avatar.server" ; pkill -f cloudflared', quiet=True)
time.sleep(1)

# No env prefix: .env.development carries it. Logs to a file so the URL is greppable.
subprocess.Popen(
    'python -m uvicorn avatar.server:app --host 0.0.0.0 --port 8000 --log-level warning'
    ' > /content/server.log 2>&1', shell=True)
subprocess.Popen(
    'cloudflared tunnel --url http://localhost:8000 --no-autoupdate'
    ' > /content/tunnel.log 2>&1', shell=True)

print('waiting for the tunnel...')
url = None
for _ in range(60):
    time.sleep(1)
    log = Path('/content/tunnel.log').read_text(errors='replace') if Path('/content/tunnel.log').exists() else ''
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', log)
    if m:
        url = m.group(0)
        break

local = sh('curl -s -m 5 http://localhost:8000/config', quiet=True).stdout
print('\nserver /config:', local or '(not responding yet)')

if not url:
    print('\nNo tunnel URL. Last 20 lines of the tunnel log:')
    print(Path('/content/tunnel.log').read_text(errors='replace')[-2000:])
else:
    print('\n' + '=' * 68)
    print('  OPEN THIS ON YOUR MAC:')
    print(f'    {url}')
    print('=' * 68)
    print('\n  1. Start session')
    print('  2. Tick "stream mic" and allow the microphone')
    print('  3. Talk, stop, and wait about a second')
    print('\n  Keep this cell running -- closing it kills the tunnel.')
    print(f'  Config check: {url}/config')


## 5. Watch it work

Run this while you are talking on your Mac. The server prints one structured JSON line per
instrumentation event, which is the same stream the page's log panel renders.


In [ ]:
# Ctrl-C / interrupt this cell when done watching.
import itertools

proc = subprocess.Popen(['tail', '-f', '/content/server.log'],
                        stdout=subprocess.PIPE, text=True, bufsize=1)
INTERESTING = ('state_change', 'latency', 'stale_dropped', 'session_failure')
try:
    for line in itertools.islice(proc.stdout, 400):
        if any(k in line for k in INTERESTING):
            print(line.rstrip())
finally:
    proc.terminate()


## 6. Keep the runtime awake

Colab reclaims an idle runtime and takes the tunnel with it. Run this in a separate cell
during a long session; it does nothing except prove the runtime is busy.


In [ ]:
for i in range(180):   # ~90 minutes
    time.sleep(30)
    if i % 10 == 0:
        alive = subprocess.run('pgrep -f "uvicorn avatar.server" >/dev/null', shell=True).returncode == 0
        print(f'{i * 30 // 60}m  server {"up" if alive else "DOWN"}', flush=True)


---

## When M0 and M2 land

Change one line in cell 3:

```python
'AVATAR_RENDERER=stub',   ->   'AVATAR_RENDERER=musetalk',
```

Plus a `.[gpu]` install and the weights, which M2 will document. Nothing else in this
notebook changes, and nothing in the server changes -- the renderer sits behind
`TalkingHeadRenderer`, and the state machine has never known which one it is talking to.
That one-line swap is the point of the whole boundary.

## What this costs you in latency

Your audio now crosses the Atlantic-or-so twice per turn. Expect the measured **3.7-5.4s**
turn from `PROCESS.md` §3.3.3 to get worse by the round trip to whichever region Colab and
Cloudflare put you in.

Worth measuring rather than guessing: compare `first frame` on the page here against the
local numbers. **Say so on the Loom** -- a demo recorded through a tunnel is not measuring
the same thing as a local one, and quietly comparing the two would be the dishonest move.
